In [2]:
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv("../mod_data.csv")

In [ ]:
df.sample(15)

,ID,text,class,target,tokens
221459,332100,How are you Hows the day been,non-suicide,0,"['How', 'are', 'you', 'Hows', 'the', 'day', 'b..."
172112,258298,AHH I just shot my shot and now I’m freaking o...,non-suicide,0,"['AHH', 'I', 'just', 'shot', 'my', 'shot', 'an..."
118048,177391,Realized todayI realized today that I am not s...,suicide,1,"['Realized', 'todayI', 'realized', 'today', 't..."
80531,120920,"Probably the wrong subreddit, but oh well.My d...",suicide,1,"['Probably', 'the', 'wrong', 'subreddit,', 'bu..."
176238,264532,I wrote a song for my girlfriend She really li...,non-suicide,0,"['I', 'wrote', 'a', 'song', 'for', 'my', 'girl..."
11148,16745,What is the point of a 650 word limit? I’m gen...,non-suicide,0,"['What', 'is', 'the', 'point', 'of', 'a', '650..."
15556,23456,"i've gone completely mental14, 4 attempts.\n\n...",suicide,1,"[""i've"", 'gone', 'completely', 'mental14,', '4..."
61115,91677,NembutalIs there a safe site where i can buy o...,suicide,1,"['NembutalIs', 'there', 'a', 'safe', 'site', '..."
215258,322835,broo i miss hong kong so bad i'm not from ther...,non-suicide,0,"['broo', 'i', 'miss', 'hong', 'kong', 'so', 'b..."
74643,112085,What keeps you going through everything?The fe...,suicide,1,"['What', 'keeps', 'you', 'going', 'through', '..."


In [ ]:
import re
from tqdm import tqdm
from functools import lru_cache
from spellchecker import SpellChecker

spell = SpellChecker()
custom_words = ['adhd', 'ptsd', 'crps', 'agender', 'lgbt', 'lgbts', 'idk', 'lol']
spell.word_frequency.load_words(custom_words)
tqdm.pandas()
# @lru_cache(maxsize=50000)
# def fix_word(word):
#     if "'" in word or len(word) <= 2:
#         return word
#     return spell.correction(word) or word

def clean_and_spellcheck(text):
    if not isinstance(text, str):
        return []
    
    # Separate glued letters and numbers (e.g., '2929\n\n\nThat's' or 'mental14')
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    
    # Normalize curly apostrophes to straight apostrophes
    text = text.replace("’", "'")
    
    # Extract clean alphabetic tokens
    clean_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    
    return clean_tokens

df['clean_tokens'] = df['text'].progress_apply(clean_and_spellcheck)

  0%|          | 81/232074 [00:26<20:52:51,  3.09it/s]


KeyboardInterrupt: 

In [ ]:
df.size()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df['clean_text'] = df['clean_tokens'].str.join(' ')

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

vectorizer = TfidfVectorizer(
    max_features=10000,   # keep top 10k terms
    ngram_range=(1, 2),   # unigrams + bigrams 
    min_df=5,             # ignore words appearing in fewer than 5 docs
    max_df=0.95,          # ignore words appearing in >95% of docs
    sublinear_tf=True     # dampen very frequent words with 1 + log(tf)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
vectorizer.vocabulary_

,ID,text,class,target,tokens,clean_tokens
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide,1,"['Ex', 'Wife', 'Threatening', 'SuicideRecently...",[]
1,3,Am I weird I don't get affected by compliments...,non-suicide,0,"['Am', 'I', 'weird', 'I', ""don't"", 'get', 'aff...",[]
2,4,Finally 2020 is almost over... So I can never ...,non-suicide,0,"['Finally', '2020', 'is', 'almost', 'over...',...",[]
3,8,i need helpjust help me im crying so hard,suicide,1,"['i', 'need', 'helpjust', 'help', 'me', 'im', ...",[]
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,1,"['I’m', 'so', 'lostHello,', 'my', 'name', 'is'...",[]
...,...,...,...,...,...,...
232069,348103,If you don't like rock then your not going to ...,non-suicide,0,"['If', 'you', ""don't"", 'like', 'rock', 'then',...",[]
232070,348106,You how you can tell i have so many friends an...,non-suicide,0,"['You', 'how', 'you', 'can', 'tell', 'i', 'hav...",[]
232071,348107,pee probably tastes like salty tea😏💦‼️ can som...,non-suicide,0,"['pee', 'probably', 'tastes', 'like', 'salty',...",[]
232072,348108,The usual stuff you find hereI'm not posting t...,suicide,1,"['The', 'usual', 'stuff', 'you', 'find', ""here...",[]
